In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("../../Tools/Predictors").resolve()))
from Predictor_Wrapper import build_batched_predictor
from enformer_pytorch import str_to_one_hot


def load_common_columns(file_paths):
    """Read tsv files, keep only their common columns, and concat."""
    dfs = [pd.read_csv(fp, sep="\t") for fp in file_paths]
    common = set(dfs[0].columns)
    for df in dfs[1:]:
        common &= set(df.columns)
    common_cols = [c for c in dfs[0].columns if c in common]  # keep original order
    return pd.concat([df[common_cols] for df in dfs], ignore_index=True)


def add_predictions(df, ckpt_list, cell_types, config,
                    length=200, batch_size=256, device_id=0, whether_softplus=True):
    """Predict with each ckpt in turn; add one functional_score_<cell> column each."""
    x = str_to_one_hot(df["Sequence"].tolist()).permute(0, 2, 1)  # (N, 4, L)
    for cell_type, ckpt in zip(cell_types, ckpt_list):
        print(f"Predicting with ckpt: {ckpt}")
        predictor = build_batched_predictor(
            model="alphagenome",
            model_config=config,
            model_ckpt=ckpt,
            length=length,
            device_id=device_id,
            batch_size=batch_size,
            whether_softplus=whether_softplus,
        )
        with torch.no_grad():
            pred = predictor(x)  # (N, 1)
        df[f"functional_score_{cell_type}"] = pred.squeeze(-1).cpu().numpy()
        del predictor
        torch.cuda.empty_cache()
    return df

/home/wangyucheng/miniconda3/envs/SeqEval_Bench/lib/python3.12/site-packages/lightning_lite/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


# DNase

In [2]:
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_offline.tsv",
]

offline_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline_alphagenome/best_loss_epoch1_loss0.2384.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline_alphagenome/best_loss_epoch0_loss0.1140.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_offline_alphagenome/best_loss_epoch0_loss0.1932.pt"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_complete.tsv", 
]

complete_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete_alphagenome/best_loss_epoch1_loss0.3319.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete_alphagenome/best_loss_epoch0_loss0.3194.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_complete_alphagenome/best_loss_epoch2_loss0.3486.pt"
]

save_dir = "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/CellType"

ALPHAGENOME_CONFIG = "/home/wangyucheng/projects/SeqEval_Bench_ExternalSource/huggingface/gtca/alphagenome_pytorch/model_all_folds.safetensors"
CELL_TYPES = ["GM12878", "HepG2", "K562"]
whether_softplus = True

os.makedirs(save_dir, exist_ok=True)

for split, paths, ckpts in [("offline", offline_path, offline_ckpt),
                            ("complete", complete_path, complete_ckpt)]:
    df = load_common_columns(paths)
    df = df.drop_duplicates(subset="Sequence").reset_index(drop=True)
    print(f"[{split}] {len(df)} unique sequences after concat + dedup")

    df = add_predictions(df, ckpts, CELL_TYPES, ALPHAGENOME_CONFIG, batch_size=1024, whether_softplus=whether_softplus)

    out_path = os.path.join(save_dir, f"DNase_{split}_starting_sequences.tsv")
    df.to_csv(out_path, sep="\t", index=False)
    print(f"Saved to: {out_path}")

/tmp/ipykernel_18466/2687791087.py:17: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(fp, sep="\t") for fp in file_paths]


[offline] 271147 unique sequences after concat + dedup
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline_alphagenome/best_loss_epoch1_loss0.2384.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline_alphagenome/best_loss_epoch1_loss0.2384.pt
[CKPT LOAD CHECK] diff = 1.724734e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline_alphagenome/best_loss_epoch1_loss0.2384.pt']
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline_alphagenome/best_loss_epoch0_loss0.1140.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline_alphagenome/best_loss_epoch0_loss0.1140.pt
[CKPT LOAD CHECK] diff = 2.204864e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline_alphagenome/best

/tmp/ipykernel_18466/2687791087.py:17: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(fp, sep="\t") for fp in file_paths]


[complete] 285653 unique sequences after concat + dedup
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete_alphagenome/best_loss_epoch1_loss0.3319.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete_alphagenome/best_loss_epoch1_loss0.3319.pt
[CKPT LOAD CHECK] diff = 1.874498e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete_alphagenome/best_loss_epoch1_loss0.3319.pt']
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete_alphagenome/best_loss_epoch0_loss0.3194.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete_alphagenome/best_loss_epoch0_loss0.3194.pt
[CKPT LOAD CHECK] diff = 2.436359e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete_alphageno

# EpisomalMPRA

In [3]:
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline.tsv",
]

offline_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline_alphagenome/best_loss_epoch2_loss0.0689.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch3_loss0.1349.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline_alphagenome/best_loss_epoch3_loss0.1675.pt"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_complete.tsv", 
]

complete_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_complete_alphagenome/best_loss_epoch4_loss0.1383.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_complete_alphagenome/best_loss_epoch1_loss0.1986.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_complete_alphagenome/best_loss_epoch1_loss0.2687.pt"
]

save_dir = "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/CellType"

ALPHAGENOME_CONFIG = "/home/wangyucheng/projects/SeqEval_Bench_ExternalSource/huggingface/gtca/alphagenome_pytorch/model_all_folds.safetensors"
CELL_TYPES = ["A549", "HepG2", "K562"]
whether_softplus = False

os.makedirs(save_dir, exist_ok=True)

for split, paths, ckpts in [("offline", offline_path, offline_ckpt),
                            ("complete", complete_path, complete_ckpt)]:
    df = load_common_columns(paths)
    df = df.drop_duplicates(subset="Sequence").reset_index(drop=True)
    print(f"[{split}] {len(df)} unique sequences after concat + dedup")

    df = add_predictions(df, ckpts, CELL_TYPES, ALPHAGENOME_CONFIG, batch_size=1024, whether_softplus=whether_softplus)

    out_path = os.path.join(save_dir, f"EpisomalMPRA_{split}_starting_sequences.tsv")
    df.to_csv(out_path, sep="\t", index=False)
    print(f"Saved to: {out_path}")

[offline] 562368 unique sequences after concat + dedup
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline_alphagenome/best_loss_epoch2_loss0.0689.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline_alphagenome/best_loss_epoch2_loss0.0689.pt
[CKPT LOAD CHECK] diff = 5.662068e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline_alphagenome/best_loss_epoch2_loss0.0689.pt']
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch3_loss0.1349.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch3_loss0.1349.pt
[CKPT LOAD CHECK] diff = 1.061925e-02
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_

# LentiMPRA

In [4]:
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_offline.tsv",
]

offline_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch18_loss0.1153.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline_alphagenome/best_loss_epoch1_loss0.0593.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_offline_alphagenome/best_loss_epoch0_loss0.4039.pt"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete.tsv", 
]

complete_ckpt = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_complete_alphagenome/best_loss_epoch19_loss0.1518.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete_alphagenome/best_loss_epoch2_loss0.0742.pt", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete_alphagenome/best_loss_epoch1_loss0.5075.pt"
]

save_dir = "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/CellType"

ALPHAGENOME_CONFIG = "/home/wangyucheng/projects/SeqEval_Bench_ExternalSource/huggingface/gtca/alphagenome_pytorch/model_all_folds.safetensors"
CELL_TYPES = ["HepG2", "K562", "WTC11"]
whether_softplus = False

os.makedirs(save_dir, exist_ok=True)

for split, paths, ckpts in [("offline", offline_path, offline_ckpt),
                            ("complete", complete_path, complete_ckpt)]:
    df = load_common_columns(paths)
    df = df.drop_duplicates(subset="Sequence").reset_index(drop=True)
    print(f"[{split}] {len(df)} unique sequences after concat + dedup")

    df = add_predictions(df, ckpts, CELL_TYPES, ALPHAGENOME_CONFIG, batch_size=1024, whether_softplus=whether_softplus)

    out_path = os.path.join(save_dir, f"LentiMPRA_{split}_starting_sequences.tsv")
    df.to_csv(out_path, sep="\t", index=False)
    print(f"Saved to: {out_path}")

[offline] 381600 unique sequences after concat + dedup
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch18_loss0.1153.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch18_loss0.1153.pt
[CKPT LOAD CHECK] diff = 9.315123e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline_alphagenome/best_loss_epoch18_loss0.1153.pt']
Predicting with ckpt: /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline_alphagenome/best_loss_epoch1_loss0.0593.pt
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline_alphagenome/best_loss_epoch1_loss0.0593.pt
[CKPT LOAD CHECK] diff = 3.602092e-03
ckpt loaded successfully from ['/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline_alphage